In [ ]:
import os
from dataclasses import dataclass

import anndata
import scanpy as sc
import torch

from trainer import Trainer, set_seed
from visualization import (
    plot_training_history,
    save_umap_plots,
    setup_plotting,
)

In [ ]:
@dataclass
class Args:
    """Configuration for training the QVAE model on single-cell data."""

    # dataset
    dataset = "immune"
    data_path = "./immune_processed.h5ad"

    # output
    output_dir = "./outputs_immune_notebook"

    # device
    device = "cuda:0"
    seed = 42

    # training
    batch_size = 512
    epochs = 100
    early_stopping_patience = 10
    checkpoint_every = 10
    val_percentage = 0.1

    # model
    hidden_dim = 512
    latent_dim = 256
    num_visible = 128
    num_hidden = 128
    normalization_method = "layer"

    # loss
    dist_beta = 10.0
    kl_beta = 1e-5

    # optimizer
    lr = 1e-4
    rbm_lr = 1e-3
    bm_weight_decay = 0.0

    # representation
    loss_type = "mse"
    representation = "q"

    # weights
    load_weights = False

    # sampler
    sampler_type = "sa"

    # SA
    sa_initial_temperature = 1000
    sa_alpha = 0.5
    sa_cutoff_temperature = 0.001
    sa_iterations_per_t = 10
    sa_size_limit = 10
    sa_rand_seed = 512

    # CIM
    project_no = "26035324"
    task_name = "demo2_qvae"
    task_mode = "sample"
    sample_number = 16
    precision = 8
    truncated_precision = 10
    target_bits = 550
    tmp_dir = "./tmp"


args = Args()

In [ ]:
device_obj = torch.device(args.device if torch.cuda.is_available() else "cpu")

set_seed(args.seed)
setup_plotting()

os.makedirs(args.output_dir, exist_ok=True)

trainer = Trainer(args, device_obj)

In [ ]:
adata = anndata.read_h5ad(args.data_path)
adata.obs_names_make_unique()

print(adata)

x = trainer.adata_to_array(adata)

batch_indices = trainer.batch_indices(adata, "batch")

train_loader, val_loader, eval_loader = trainer.make_loaders(x, batch_indices)

In [ ]:
model, vae_optimizer, bm_optimizer = trainer.create_model(adata.n_vars, train_loader)

In [ ]:
weights_path = os.path.join(args.output_dir, "immune_kpp_qvae_best.pth")

history = trainer.train(
    model, train_loader, val_loader, vae_optimizer, bm_optimizer, weights_path
)

In [ ]:
plot_training_history(history, os.path.join(args.output_dir, "training_curves.png"))

In [ ]:
reps = trainer.extract_representation(model, eval_loader)

adata.obsm["X_qvae"] = reps

In [ ]:
sc.pp.neighbors(adata, use_rep="X_qvae", n_neighbors=15)

sc.tl.umap(adata, random_state=args.seed)

save_umap_plots(adata, "final_annotation", "batch", args.output_dir)